In [ ]:
# Чтобы не выскакивали всякие бесполезные предупреждения
import warnings
warnings.filterwarnings('ignore')

Устанавливаем библиотеки для дообучения:

In [ ]:
!pip install -U bitsandbytes accelerate transformers trl==0.19.0 datasets peft

#Добавление данных в датасет


In [ ]:
import pandas as pd
data = pd.read_csv('english_data_10.csv')

In [ ]:
data.tail(10)

,prompt,response
905,Draw a very dark gray circle without animation,from manim import *\n\nclass VeryDarkGrayCircl...
906,Draw a darker gray circle without animation,from manim import *\n\nclass DarkerGrayCircle(...
907,Draw a very dark grey circle without animation,from manim import *\n\nclass VeryDarkGreyCircl...
908,Draw a darker grey circle without animation,from manim import *\n\nclass DarkerGreyCircle(...
909,Draw a grey brown circle without animation,from manim import *\n\nclass GreyBrownCircle(S...
910,Draw a gray brown circle without animation,from manim import *\n\nclass GrayBrownCircle(S...
911,Draw a white circle without animation,from manim import *\n\nclass WhiteCircle(Scene...
912,Draw a manim's logo white circle without anima...,from manim import *\n\nclass ManimsLogoWhiteCi...
913,Draw a pastel white circle without animation,from manim import *\n\nclass PastelWhiteCircle...
914,Create an animation where three squares appear...,from manim import *\n\nclass Squares(Scene):\n...


Изменяем конкретную(i-ую) строчку:

In [ ]:
i = 359

row = data.iloc[i]

print(row['prompt'])
print()
print(row['response'])

Display the phrase "fl ligature" twice in large font, stacked vertically with some spacing. The top version uses a typographic ligature for "fl", while the bottom version disables ligatures, showing the "f" and "l" as separate characters.

from manim import *

class DisableLigature(Scene):
    def construct(self):
        li = Text("fl ligature",font_size=96)
        nli = Text("fl ligature", disable_ligatures=True, font_size=96)
        self.add(Group(li, nli).arrange(DOWN, buff=.8))


In [ ]:
# Задаём новые значения: [промпт, релевантный код]
data.loc[i] = [prow['prompt'], """from manim import *\nclass NotoSansText(Scene):\n    def construct(self):\n        ft = Text("Noto Sans", font="Noto Sans")\n        self.add(ft)
"""]

Добавляем новую строку:

In [ ]:
new_row = {
    'prompt':"""Display a text label containing the monospaced letter H with a looped arrow and the LaTeX logo. The font size is 144.""",
    'response':"""from manim import *\n\nclass AMSLaTeX(Scene):\n    def construct(self):\n        tex = Tex(r'$\mathtt{H} \looparrowright$ \LaTeX', font_size=144)\n        self.add(tex)"""
    }

data.loc[len(data)] = new_row

<>:3: SyntaxWarning: invalid escape sequence '\m'
<>:3: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipython-input-2408432524.py:3: SyntaxWarning: invalid escape sequence '\m'
  'response':"""from manim import *\n\nclass AMSLaTeX(Scene):\n    def construct(self):\n        tex = Tex(r'$\mathtt{H} \looparrowright$ \LaTeX', font_size=144)\n        self.add(tex)"""


Сохраняем изменненый датасет:

In [ ]:
data.to_csv('english_data_9.csv', index=False)

# Обучение маленьких моделей

Загружаем датасет:

In [ ]:
import pandas as pd

df = pd.read_csv("english_data_12.csv")

Загружаем начальную модель, которую будем обучать (в нашем случае Qwen2.5-Coder-3B-Instruct)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

default_model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
default_tokenizer = AutoTokenizer.from_pretrained(default_model_name)
default_model = AutoModelForCausalLM.from_pretrained(
    default_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Работаем с данными:

In [ ]:
from datasets import Dataset

# Формируем инструкции, на которых будем дообучать модель
def format_instruction(sample):
    text = (
        "<|im_start|>system\n"
        "Write ONLY the code (without text explanations and comments) using the manim library for Python, which corresponds to the user's request."
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{sample['prompt']}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{sample['response']}"
        "<|im_end|>"
    )
    return {"text": text}

train_dataset = Dataset.from_pandas(df[["prompt", "response"]])
train_dataset = train_dataset.map(
    format_instruction,
    remove_columns=["prompt", "response"]
)

Map:   0%|          | 0/913 [00:00<?, ? examples/s]

In [ ]:
# Разбиваем выборку на тренировочную и тестовую
train_test = train_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test["train"]
test_dataset = train_test["test"]
test_dataset = test_dataset.filter(lambda ex: len(default_tokenizer.encode(ex['text'])) >= 2)

Filter:   0%|          | 0/183 [00:00<?, ? examples/s]

In [ ]:
# Отфильтровываем слишком длинные инструкции
def filter_long(example):
    tokens = default_tokenizer(example["text"], truncation=True, max_length=1024)
    decoded = default_tokenizer.decode(tokens["input_ids"])
    return "<|im_start|>assistant\n" in decoded

test_dataset = test_dataset.filter(filter_long)

Filter:   0%|          | 0/183 [00:00<?, ? examples/s]

Настраиваем модель:

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=6,
    lora_alpha=12
)

model = get_peft_model(default_model, peft_config)

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

data_collator = DataCollatorForCompletionOnlyLM(
    response_template = "<|im_start|>assistant\n",
    tokenizer = default_tokenizer,
)

In [ ]:
from trl import SFTConfig

torch.cuda.empty_cache()

training_args = SFTConfig(
    bf16=False,
    fp16=False,

    output_dir="./results",
    num_train_epochs=7,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    dataset_text_field="text",

    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    save_steps=250,
    save_total_limit=5,

    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    prediction_loss_only=False,

    max_length=1024
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    data_collator = data_collator,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset = test_dataset,
)

Adding EOS to train dataset:   0%|          | 0/730 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/730 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/730 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/183 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/183 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/183 [00:00<?, ? examples/s]

Обучаем:

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.763070,0.664380
100,0.686139,0.617855
150,0.635310,0.592750
200,0.632104,0.583900


Step,Training Loss,Validation Loss
50,0.763070,0.664380
100,0.686139,0.617855
150,0.635310,0.592750
200,0.632104,0.583900
250,0.567950,0.565285
300,0.562895,0.556277
350,0.522712,0.551810
400,0.528224,0.548864
450,0.495459,0.549641
500,0.460330,0.545810


TrainOutput(global_step=644, training_loss=0.5607716607751313, metrics={'train_runtime': 9000.1497, 'train_samples_per_second': 0.568, 'train_steps_per_second': 0.072, 'total_flos': 2.208213464174592e+16, 'train_loss': 0.5607716607751313})

Сохраняем дообученную модель:

In [ ]:
# Сохраняем (не забыть затем сохранить в архив и скачать!)
trainer.save_model('./tune_model_english_qwen_coder')

#Работа с архивами

Сохраняем модель в архив:

In [ ]:
import shutil

shutil.make_archive("tune_model_english_qwen_coder", 'zip', "./tune_model_english_qwen_coder")

'/content/tune_model_english_qwen_coder_3b_16_03_26.zip'

Распаковываем модель из архива:

In [ ]:
import zipfile

with zipfile.ZipFile('tune_model_english_qwen_coder.zip', 'r') as zip_ref:
    zip_ref.extractall('tune_model_english_qwen_coder/')

#Для проверки manim-кода

**Отдельный [блокнот](https://colab.research.google.com/drive/1ODyxxgHIDes4U5LArJ8PCK1GL1P7S7rD?usp=sharing) для теста manim-кода**

#Для оценки крутости модели

Загружаем модель:

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "./tune_model_english_qwen_coder"

test_bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  # load_in_8bit=True,
  # llm_int8_enable_fp32_cpu_offload=True,
)

test_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    # quantization_config=test_bnb_config,
    low_cpu_mem_usage=True
)
test_tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

Функция для тестирования кода на возникновение ошибок при запуске:

In [ ]:
import subprocess
import tempfile
import os
import re
import traceback

# Функция тестирует код на возникновение ошибки при запуске
# Если ошибка возникает, то она логгируется в manim_errors.log
def manim_test(idx: int, code_string: str, log_file: str = "manim_errors.log") -> bool:
    match_text = re.search(r'class\s+(\w+)\s*\(', code_string)
    if match_text is None:
        with open(log_file, 'a', encoding='utf-8') as log:
          log.write("--- Failed to find class ---\n")
          log.write(f"IDX:\n{idx}\n")
          log.write("--- End ---\n\n")
        return False
    class_name = match_text.group(1)

    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(code_string)
        temp_file = f.name

    try:
        result = subprocess.run(
            ["manim", "-ql", "--disable_caching", temp_file, class_name],
            capture_output=True,
            text=True,
            timeout=10
        )

        if result.returncode != 0:
            with open(log_file, 'a', encoding='utf-8') as log:
                log.write("--- Manim error ---\n")
                log.write(f"IDX:\n{idx}\n")
                log.write(f"STDERR:\n{result.stderr}\n")
                log.write(f"STDOUT:\n{result.stdout}\n")
                log.write("--- End ---\n\n")
            return False

        return True

    except Exception as e:
        with open(log_file, 'a', encoding='utf-8') as log:
            log.write("--- Exception during Manim execution ---\n")
            log.write(f"Code:\n{code_string}\n")
            log.write(f"Error: {str(e)}\n")
            log.write(traceback.format_exc())
            log.write("--- End ---\n\n")
        return False

    finally:
        if os.path.exists(temp_file):
            os.unlink(temp_file)

In [ ]:
import pandas as pd
data = pd.read_csv('english_data_12.csv')

Устанавливаем библиотеки для работы с manim:

In [ ]:
!sudo apt update
!sudo apt install libcairo2-dev \
    texlive texlive-latex-extra texlive-fonts-extra \
    texlive-latex-recommended texlive-science \
    tipa libpango1.0-dev
!pip install manim
!pip install IPython==8.21.0

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
132 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

In [ ]:
import time

n = 100 # Размер случайной выборки из датасета
random_samples = data.sample(n=n, random_state=42)

temperature = 0.8

failed = [] # Номера провваленых тестов

for i, (idx, row) in enumerate(random_samples.iterrows()):
    torch.cuda.empty_cache()

    messages = [
        {"role": "system", "content": "Write ONLY the code (without text explanations and comments) using the manim library for Python, which corresponds to the user's request."},
        {"role": "user", "content": row['prompt']}
    ]

    text = test_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = test_tokenizer(text, return_tensors="pt").to(test_model.device)
    outputs = test_model.generate(
        **inputs,
        temperature=temperature,
        repetition_penalty=1.2,
        max_new_tokens=500
    )

    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    code = test_tokenizer.decode(generated_tokens, skip_special_tokens=True)

    if not manim_test(idx, code):
        failed.append(idx)

    if (i + 1) % 10 == 0:
        print(f"[{time.strftime('%H:%M:%S')} GMT] Обработано {i + 1}/{n}. Неуспешных попыток: {len(failed)}")

[10:03:28 GMT] Обработано 10/100. Неуспешных попыток: 8
[10:05:07 GMT] Обработано 20/100. Неуспешных попыток: 15
[10:07:00 GMT] Обработано 30/100. Неуспешных попыток: 23
[10:09:52 GMT] Обработано 40/100. Неуспешных попыток: 33
[10:11:44 GMT] Обработано 50/100. Неуспешных попыток: 42
[10:13:47 GMT] Обработано 60/100. Неуспешных попыток: 49
[10:16:19 GMT] Обработано 70/100. Неуспешных попыток: 57
[10:17:49 GMT] Обработано 80/100. Неуспешных попыток: 64
[10:19:43 GMT] Обработано 90/100. Неуспешных попыток: 71
[10:22:11 GMT] Обработано 100/100. Неуспешных попыток: 76


In [ ]:
# Температура модели
print(f"Temperature: {temperature}")

# Количество тестов из датасета, для которых модель написала работающий код
print(f'Accuracy: {(n - len(failed)) / n:.4f}')

# Тесты для которых модель написала неработающую ерунду
print('Failed: ', *failed)

Temperature: 0.8
Accuracy: 0.2400
Failed:  380 355 357 362 486 595 551 30 590 521 215 63 39 424 72 908 168 621 70 67 306 718 638 321 784 548 342 275 501 294 499 630 23 440 411 218 522 613 911 754 604 743 363 578 107 539 467 312 444 139 141 239 311 789 44 250 542 824 247 292 800 589 323 876 60 209 417 86 66 331 110 227 265 575 394 76
